<a href="https://colab.research.google.com/github/CathalD/Community_Carbon_Analysis_and_Mapping/blob/Transfer_Oct/GEE_Raster_Exports.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%pip install -q earthengine-api geemap

In [6]:
# ==================================================
# Step 1 — Settings and connection
# ==================================================

import ee
import geemap
import json
from google.colab import files

# ---- User settings ----

gee_project = "northstarlabs"

analysis_scale = "local"       # local, regional, broad
grid_size_m = 30
context_radii_m = [120, 1000]

start_year = 2019
end_year = 2023

output_crs = "EPSG:3978"       # Same coordinate system as Script2

# Start with topography; add other groups later.
covariate_groups = ["topography"]

# ---- Connect to Earth Engine ----

ee.Authenticate()
ee.Initialize(project=gee_project)

In [13]:
# ==================================================
# Step 2 — AOI and output grid
# ==================================================

# ---- Upload the boundary ----

uploaded = files.upload()

aoi_filename = next(iter(uploaded))
aoi_data = json.loads(uploaded[aoi_filename])

aoi_features = ee.FeatureCollection(aoi_data)
aoi = aoi_features.geometry()

# ---- Define the output grid ----
# A fixed origin keeps future raster groups aligned.

grid_transform = [
    grid_size_m, 0, 0,
    0, -grid_size_m, 0
]

grid_projection = ee.Projection(
    output_crs,
    grid_transform
)

# ---- Define dates ----
# Earth Engine excludes the end date.

start_date = f"{start_year}-01-01"
end_date = f"{end_year + 1}-01-01"

# ---- Inspect the settings ----

aoi_area_km2 = aoi.area(maxError=1).divide(1e6).getInfo()

print("AOI file:", aoi_filename)
print("AOI area:", round(aoi_area_km2, 2), "km²")
print("Analysis scale:", analysis_scale)
print("Grid size:", grid_size_m, "m")
print("Context radii:", context_radii_m, "m")
print("Coordinate system:", output_crs)
print("Years:", start_year, "to", end_year)

# ---- Display the boundary ----

import geemap

Map = geemap.Map(basemap="Esri.WorldImagery")
Map.centerObject(aoi)

boundary = aoi_features.style(
    color="red",
    fillColor="00000000",
    width=2
)

Map.addLayer(boundary, {}, "AOI boundary")

display(Map)

Saving aoi.geojson to aoi (2).geojson
AOI file: aoi (2).geojson
AOI area: 13578.08 km²
Analysis scale: local
Grid size: 30 m
Context radii: [120, 1000] m
Coordinate system: EPSG:3978
Years: 2019 to 2023


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [14]:
# ==================================================
# Step 3 — Import and inspect elevation
# ==================================================

# ---- Include context around the AOI ----
# Keep this buffer for later neighbourhood calculations.

buffer_m = max(context_radii_m) + grid_size_m
processing_area = aoi.buffer(buffer_m)

# ---- Load the elevation tiles ----

dem_collection = (
    ee.ImageCollection("NRCan/CDEM")
    .filterBounds(processing_area)
    .select("elevation")
)

tile_count = dem_collection.size().getInfo()
print("Elevation tiles found:", tile_count)

if tile_count == 0:
    raise ValueError("No CDEM tiles found for this AOI.")

# ---- Combine the tiles ----
# Preserve the source projection when creating the mosaic.

dem_first = ee.Image(dem_collection.first())

dem = (
    dem_collection.mosaic()
    .setDefaultProjection(dem_first.projection())
    .rename("elevation_m")
    .clip(processing_area)
)

# ---- Get an approximate range for map colours ----
# This 300 m sampling is for the preview only.

dem_stats = dem.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi,
    crs=output_crs,
    scale=300,
    maxPixels=1e7
).getInfo()

elevation_min = dem_stats["elevation_m_min"]
elevation_max = dem_stats["elevation_m_max"]

print("Preview minimum elevation:", elevation_min, "m")
print("Preview maximum elevation:", elevation_max, "m")

# ---- Display elevation inside the AOI ----

dem_vis = {
    "min": elevation_min,
    "max": elevation_max,
    "palette": ["2166ac", "67a9cf", "d9ef8b", "a6611a", "ffffff"]
}

Map.addLayer(dem.clip(aoi), dem_vis, "Elevation (m)")

# Draw the boundary above the elevation layer.
Map.addLayer(boundary, {}, "AOI boundary")

display(Map)

Elevation tiles found: 5
Preview minimum elevation: 261 m
Preview maximum elevation: 617 m


Map(bottom=11628.0, center=[48.45835188280866, -83.67187500000001], controls=(WidgetControl(options=['position…

In [15]:
# ==================================================
# Step 4 — Slope and relative elevation
# ==================================================

# ---- Prepare the terrain working grid ----

terrain_cell_m = 30

terrain_projection = ee.Projection(output_crs).atScale(
    terrain_cell_m
)

dem_terrain = (
    dem.resample("bilinear")
    .reproject(terrain_projection)
)

# ---- Calculate slope ----

slope = (
    ee.Terrain.slope(dem_terrain)
    .rename("slope_deg")
    .reproject(terrain_projection)
)

Map.addLayer(
    slope.clip(aoi),
    {
        "min": 0,
        "max": 15,
        "palette": ["ffffff", "a1d99b", "006d2c"]
    },
    "Slope (degrees)"
)

# ---- Start the topography group ----

topography = dem_terrain.addBands(slope)

# ---- Calculate relative elevation at each radius ----

for radius in context_radii_m:

    # Convert the radius from metres to working-grid cells.
    kernel = ee.Kernel.circle(
        radius=radius / terrain_cell_m,
        units="pixels"
    )

    neighbourhood_mean = (
        dem_terrain.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel
        )
        .reproject(terrain_projection)
    )

    relative_elevation = (
        dem_terrain.subtract(neighbourhood_mean)
        .rename(f"relative_elevation_{radius}m")
    )

    topography = topography.addBands(relative_elevation)

    Map.addLayer(
        relative_elevation.clip(aoi),
        {
            "min": -10,
            "max": 10,
            "palette": ["2166ac", "f7f7f7", "b2182b"]
        },
        f"Relative elevation — {radius} m radius",
        False
    )

# ---- Restrict the completed group to the AOI ----

topography = topography.clip(aoi)

print("Topography bands:", topography.bandNames().getInfo())

# ---- Display the updated map ----

Map.addLayer(boundary, {}, "AOI boundary")

display(Map)

Topography bands: ['elevation_m', 'slope_deg', 'relative_elevation_120m', 'relative_elevation_1000m']


Map(bottom=22956.0, center=[48.45835188280866, -83.67187500000001], controls=(WidgetControl(options=['position…

In [16]:
%pip install -q "geedim>=2.0.1,<3" rasterio

# ==================================================
# Shared settings
# ==================================================

working_cell_m = terrain_cell_m
working_projection = terrain_projection

cloud_percent = 40
minimum_observations = 5
radar_pass = "DESCENDING"
water_distance_max_m = 12000

median_reducer = ee.Reducer.median(maxRaw=10000)

# Two helpers used by the groups below.

def to_working_grid(image, native_projection, reducer):
    return (
        image.setDefaultProjection(native_projection)
        .reduceResolution(reducer=reducer, maxPixels=4096)
        .reproject(working_projection)
    )

def focal_summary(image, radius, reducer):
    kernel = ee.Kernel.circle(
        radius / working_cell_m, "pixels"
    )
    return (
        image.reduceNeighborhood(
            reducer=reducer, kernel=kernel
        )
        .reproject(working_projection)
    )

In [17]:
# ==================================================
# Topography — add neighbourhood relief
# ==================================================

for radius in context_radii_m:

    relief = focal_summary(
        dem_terrain, radius, ee.Reducer.stdDev()
    )

    topography = topography.addBands(
        relief.rename(f"elevation_{radius}m_spatial_sd"),
        overwrite=True
    )

topography = topography.clip(aoi)

print("Topography:", topography.bandNames().getInfo())

Topography: ['elevation_m', 'slope_deg', 'relative_elevation_120m', 'relative_elevation_1000m', 'elevation_120m_spatial_sd', 'elevation_1000m_spatial_sd']


In [18]:
# ==================================================
# Vegetation and moisture
# ==================================================

def prepare_s2(image):

    scl = image.select("SCL")
    clear = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6))

    reflectance = (
        image.select(["B2", "B4", "B8", "B11"])
        .multiply(0.0001)
        .updateMask(clear)
        .resample("bilinear")
    )

    ndvi = reflectance.normalizedDifference(
        ["B8", "B4"]
    ).rename("NDVI")

    ndmi = reflectance.normalizedDifference(
        ["B8", "B11"]
    ).rename("NDMI")

    evi = reflectance.expression(
        "2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1)",
        {
            "nir": reflectance.select("B8"),
            "red": reflectance.select("B4"),
            "blue": reflectance.select("B2")
        }
    ).rename("EVI")

    return (
        ndvi.addBands([ndmi, evi])
        .copyProperties(image, ["system:time_start"])
    )

# ---- Load and prepare images ----

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(processing_area)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lte(
        "CLOUDY_PIXEL_PERCENTAGE", cloud_percent
    ))
    .map(prepare_s2)
)

s2_spring = s2.filter(
    ee.Filter.calendarRange(4, 6, "month")
)

s2_summer = s2.filter(
    ee.Filter.calendarRange(6, 8, "month")
)

s2_projection = (
    ee.Image(s2.first()).select("NDVI").projection()
)

# ---- Seasonal summaries ----

vegetation_native = (
    s2_spring.select("NDMI")
    .reduce(median_reducer)
    .rename("ndmi_spring_median")

    .addBands(
        s2_spring.select("NDMI")
        .reduce(ee.Reducer.stdDev())
        .rename("ndmi_spring_temporal_sd")
    )

    .addBands(
        s2_summer.select("NDVI")
        .reduce(median_reducer)
        .rename("ndvi_summer_median")
    )

    .addBands(
        s2_summer.select("NDVI")
        .reduce(ee.Reducer.stdDev())
        .rename("ndvi_summer_temporal_sd")
    )

    .addBands(
        s2_summer.select("EVI")
        .reduce(median_reducer)
        .rename("evi_summer_median")
    )

    .addBands(
        s2_summer.select("EVI")
        .reduce(ee.Reducer.stdDev())
        .rename("evi_summer_temporal_sd")
    )
)

vegetation = to_working_grid(
    vegetation_native, s2_projection, ee.Reducer.mean()
)

# ---- Observation counts ----

s2_spring_count = to_working_grid(
    s2_spring.select("NDMI").count().unmask(0),
    s2_projection,
    ee.Reducer.min()
).rename("s2_spring_count")

s2_summer_count = to_working_grid(
    s2_summer.select("NDVI").count().unmask(0),
    s2_projection,
    ee.Reducer.min()
).rename("s2_summer_count")

vegetation = vegetation.updateMask(
    s2_spring_count.gte(minimum_observations)
    .And(s2_summer_count.gte(minimum_observations))
)

# ---- Neighbourhood summaries ----

for radius in context_radii_m:

    base = vegetation.select("ndvi_summer_median")

    vegetation = vegetation.addBands([
        focal_summary(base, radius, ee.Reducer.mean())
        .rename(f"ndvi_{radius}m_mean"),

        focal_summary(base, radius, ee.Reducer.stdDev())
        .rename(f"ndvi_{radius}m_spatial_sd")
    ])

vegetation = vegetation.clip(aoi)

print("Vegetation:", vegetation.bandNames().getInfo())

Vegetation: ['ndmi_spring_median', 'ndmi_spring_temporal_sd', 'ndvi_summer_median', 'ndvi_summer_temporal_sd', 'evi_summer_median', 'evi_summer_temporal_sd', 'ndvi_120m_mean', 'ndvi_120m_spatial_sd', 'ndvi_1000m_mean', 'ndvi_1000m_spatial_sd']


In [19]:
# ==================================================
# Land cover — Dynamic World
# ==================================================

dw = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(processing_area)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.calendarRange(6, 8, "month"))
)

dw_projection = (
    ee.Image(dw.first()).select("trees").projection()
)

probability_bands = [
    "trees", "shrub_and_scrub",
    "flooded_vegetation", "water"
]

probability_names = [
    "tree", "shrub", "floodedveg", "water"
]

# ---- Mean probabilities and temporal variability ----

dw_mean = to_working_grid(
    dw.select(probability_bands).mean(),
    dw_projection,
    ee.Reducer.mean()
).rename([
    f"dw_{name}_prob_mean"
    for name in probability_names
])

dw_sd = to_working_grid(
    dw.select(probability_bands).reduce(
        ee.Reducer.stdDev()
    ),
    dw_projection,
    ee.Reducer.mean()
).rename([
    f"dw_{name}_prob_temporal_sd"
    for name in probability_names
])

dw_count = to_working_grid(
    dw.select("trees").count().unmask(0),
    dw_projection,
    ee.Reducer.min()
).rename("dynamic_world_count")

dw_mean = dw_mean.updateMask(
    dw_count.gte(minimum_observations)
)

land_cover = dw_mean.addBands(dw_sd).updateMask(
    dw_count.gte(minimum_observations)
)

# ---- Neighbourhood probabilities ----

for radius in context_radii_m:

    context = focal_summary(
        dw_mean, radius, ee.Reducer.mean()
    )

    land_cover = land_cover.addBands(
        context.rename([
            f"dw_{name}_{radius}m_mean"
            for name in probability_names
        ])
    )

# ---- Most frequent class ----
# Use mode, not an average of class numbers.

dw_class = to_working_grid(
    dw.select("label").mode(),
    dw_projection,
    ee.Reducer.mode()
).rename("dw_class_mode").updateMask(
    dw_count.gte(minimum_observations)
)

land_cover = land_cover.addBands(dw_class).clip(aoi)

print("Land cover:", land_cover.bandNames().getInfo())

Land cover: ['dw_tree_prob_mean', 'dw_shrub_prob_mean', 'dw_floodedveg_prob_mean', 'dw_water_prob_mean', 'dw_tree_prob_temporal_sd', 'dw_shrub_prob_temporal_sd', 'dw_floodedveg_prob_temporal_sd', 'dw_water_prob_temporal_sd', 'dw_tree_120m_mean', 'dw_shrub_120m_mean', 'dw_floodedveg_120m_mean', 'dw_water_120m_mean', 'dw_tree_1000m_mean', 'dw_shrub_1000m_mean', 'dw_floodedveg_1000m_mean', 'dw_water_1000m_mean', 'dw_class_mode']


In [24]:
# ==================================================
# Radar — Sentinel-1
# ==================================================

def prepare_s1(image):

    contrast = (
        image.select("VV").subtract(image.select("VH"))
    )

    # Convert the dB difference to a linear VV/VH ratio.
    ratio = ee.Image.constant(10).pow(
        contrast.divide(10)
    )

    return (
        image.select(["VV", "VH"])
        .addBands(contrast.rename("contrast"))
        .addBands(ratio.rename("ratio"))
        .copyProperties(image, ["system:time_start"])
    )

# ---- Check radar availability before selecting an image ----

s1_available = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(processing_area)
    .filterDate(start_date, end_date)
)

s1_base = (
    s1_available
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains(
        "transmitterReceiverPolarisation", "VV"
    ))
    .filter(ee.Filter.listContains(
        "transmitterReceiverPolarisation", "VH"
    ))
)

pass_counts = ee.Dictionary({
    "ASCENDING": s1_base.filter(
        ee.Filter.eq("orbitProperties_pass", "ASCENDING")
    ).size(),

    "DESCENDING": s1_base.filter(
        ee.Filter.eq("orbitProperties_pass", "DESCENDING")
    ).size()
}).getInfo()

print("Available IW VV/VH images:", pass_counts)

# ---- Use the requested pass, or the other available pass ----

requested_radar_pass = radar_pass

if radar_pass not in pass_counts:
    raise ValueError(
        "radar_pass must be ASCENDING or DESCENDING."
    )

if pass_counts[radar_pass] == 0:

    other_pass = (
        "ASCENDING" if radar_pass == "DESCENDING"
        else "DESCENDING"
    )

    if pass_counts[other_pass] == 0:

        print(
            "Available modes:",
            s1_available.aggregate_histogram(
                "instrumentMode"
            ).getInfo()
        )

        print(
            "Available polarisations:",
            s1_available.aggregate_array(
                "transmitterReceiverPolarisation"
            ).distinct().getInfo()
        )

        raise ValueError(
            "No IW VV/VH images match this AOI and date range."
        )

    radar_pass = other_pass

    print(
        f"No {requested_radar_pass} images; "
        f"using {radar_pass} only."
    )

# ---- Prepare the selected images ----

s1 = (
    s1_base
    .filter(ee.Filter.eq(
        "orbitProperties_pass", radar_pass
    ))
    .map(prepare_s1)
)

print("Radar pass used:", radar_pass)

s1_projection = (
    ee.Image(s1.first()).select("VV").projection()
)

radar_names = [
    "vv_db", "vh_db",
    "vv_minus_vh_db", "vv_over_vh_linear"
]

radar_native = (
    s1.reduce(median_reducer)
    .rename([f"{name}_median" for name in radar_names])

    .addBands(
        s1.reduce(ee.Reducer.stdDev()).rename([
            f"{name}_temporal_sd" for name in radar_names
        ])
    )
)

radar = to_working_grid(
    radar_native, s1_projection, ee.Reducer.mean()
)

s1_count = to_working_grid(
    s1.select("VV").count().unmask(0),
    s1_projection,
    ee.Reducer.min()
).rename("sentinel1_count")

radar = radar.updateMask(
    s1_count.gte(minimum_observations)
).clip(aoi)

print("Radar:", radar.bandNames().getInfo())

Available IW VV/VH images: {'ASCENDING': 642, 'DESCENDING': 0}
No DESCENDING images; using ASCENDING only.
Radar pass used: ASCENDING
Radar: ['vv_db_median', 'vh_db_median', 'vv_minus_vh_db_median', 'vv_over_vh_linear_median', 'vv_db_temporal_sd', 'vh_db_temporal_sd', 'vv_minus_vh_db_temporal_sd', 'vv_over_vh_linear_temporal_sd']


In [25]:
# ==================================================
# Historical surface water
# ==================================================

surface_water = ee.Image(
    "JRC/GSW1_4/GlobalSurfaceWater"
)

water_projection = (
    surface_water.select("occurrence").projection()
)

water_native = surface_water.select([
    "occurrence", "seasonality", "recurrence"
]).unmask(0).rename([
    "water_occurrence_pct",
    "water_seasonality_months",
    "water_recurrence_pct"
])

water = to_working_grid(
    water_native, water_projection, ee.Reducer.mean()
)

# Keep the source outside the AOI for distance calculations.

observed_water = (
    surface_water.select("max_extent")
    .eq(1)
    .unmask(0)
    .reproject(working_projection)
)

water_distance = (
    observed_water.fastDistanceTransform(
        neighborhood=int(
            water_distance_max_m / working_cell_m
        ),
        units="pixels",
        metric="squared_euclidean"
    )
    .sqrt()
    .multiply(working_cell_m)
    .min(water_distance_max_m)
    .unmask(water_distance_max_m)
    .rename("observed_water_distance_m")
    .reproject(working_projection)
)

water = water.addBands(water_distance).clip(aoi)

print("Water:", water.bandNames().getInfo())

Water: ['water_occurrence_pct', 'water_seasonality_months', 'water_recurrence_pct', 'observed_water_distance_m']


In [26]:
# ==================================================
# Climate — ERA5-Land
# ==================================================

era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterBounds(aoi)
    .filterDate(start_date, end_date)
)

expected_months = (end_year - start_year + 1) * 12

if era5.size().getInfo() != expected_months:
    raise ValueError(
        "Choose complete calendar years for the climate summaries."
    )

climate_projection = (
    ee.Image(era5.first())
    .select("temperature_2m")
    .projection()
)

# ---- Build annual summaries ----

annual_precipitation = []
annual_freezing = []

for year in range(start_year, end_year + 1):

    yearly = era5.filterDate(
        f"{year}-01-01", f"{year + 1}-01-01"
    )

    annual_precipitation.append(
        yearly.select("total_precipitation_sum")
        .sum()
        .multiply(1000)
        .rename("annual_precip_mm")
    )

    annual_freezing.append(
        yearly.select("temperature_2m")
        .map(lambda image: image.lt(273.15))
        .sum()
        .rename("freezing_months")
    )

annual_precipitation = ee.ImageCollection.fromImages(
    annual_precipitation
)

annual_freezing = ee.ImageCollection.fromImages(
    annual_freezing
)

# ---- Combine climate descriptors ----

climate = (
    era5.select("temperature_2m")
    .mean()
    .subtract(273.15)
    .rename("mean_temperature_c")

    .addBands(
        era5.select("temperature_2m")
        .reduce(ee.Reducer.stdDev())
        .rename("monthly_temperature_sd_c")
    )

    .addBands(
        annual_precipitation.mean()
        .rename("mean_annual_precip_mm")
    )

    .addBands(
        annual_precipitation.reduce(ee.Reducer.stdDev())
        .rename("annual_precip_sd_mm")
    )

    .addBands(
        annual_freezing.mean()
        .rename("mean_freezing_months")
    )

    .setDefaultProjection(climate_projection)
    .clip(aoi)
)

print("Climate:", climate.bandNames().getInfo())

Climate: ['mean_temperature_c', 'monthly_temperature_sd_c', 'mean_annual_precip_mm', 'annual_precip_sd_mm', 'mean_freezing_months']


In [27]:
# ==================================================
# Quality — keep separate from predictors
# ==================================================

quality = (
    s2_spring_count
    .addBands(s2_summer_count)
    .addBands(dw_count)
    .addBands(s1_count)
    .clip(aoi)
)

print("Quality:", quality.bandNames().getInfo())

Quality: ['s2_spring_count', 's2_summer_count', 'dynamic_world_count', 'sentinel1_count']


In [ ]:
# ==================================================
# Export — one multi-band TIFF per group
# ==================================================

import geedim
import rasterio
import csv
import math
import shutil
from pathlib import Path
from datetime import datetime
from google.colab import files

# ---- Export settings ----

export_cell_m = grid_size_m
nodata_value = -9999

export_groups = {
    "topography": topography,
    "vegetation": vegetation,
    "land_cover": land_cover,
    "radar": radar,
    "water": water,
    "climate": climate,
    "quality": quality
}

sources = {
    "topography": "NRCan/CDEM",
    "vegetation": "COPERNICUS/S2_SR_HARMONIZED",
    "land_cover": "GOOGLE/DYNAMICWORLD/V1",
    "radar": "COPERNICUS/S1_GRD",
    "water": "JRC/GSW1_4/GlobalSurfaceWater",
    "climate": "ECMWF/ERA5_LAND/MONTHLY_AGGR",
    "quality": "Sentinel-2, Dynamic World and Sentinel-1 counts"
}

# ---- Establish identical bounds for every file ----

ring = (
    aoi.bounds(maxError=1, proj=output_crs)
    .coordinates().getInfo()[0]
)

left = math.floor(
    min(p[0] for p in ring) / export_cell_m
) * export_cell_m

right = math.ceil(
    max(p[0] for p in ring) / export_cell_m
) * export_cell_m

bottom = math.floor(
    min(p[1] for p in ring) / export_cell_m
) * export_cell_m

top = math.ceil(
    max(p[1] for p in ring) / export_cell_m
) * export_cell_m

shape = (
    round((top - bottom) / export_cell_m),
    round((right - left) / export_cell_m)
)

export_transform = [
    export_cell_m, 0, left,
    0, -export_cell_m, top
]

export_projection = ee.Projection(
    output_crs, export_transform
)

# ---- Create a new run folder in Colab ----

run_name = (
    f"covariates_{analysis_scale}_{export_cell_m}m_"
    f"{datetime.now():%Y%m%d_%H%M%S}"
)

output_folder = Path("/content") / run_name
output_folder.mkdir()

manifest = []

# ---- Download each group ----

for group_name, image in export_groups.items():

    band_names = image.bandNames().getInfo()

    aggregation = (
        "nearest neighbour; native climate support retained"
    )

    # Aggregate fine layers when exporting larger cells.
    if (
        export_cell_m > working_cell_m
        and group_name != "climate"
    ):

        aggregation = "mean of valid working-grid cells"

        if group_name == "quality":

            image = image.reduceResolution(
                ee.Reducer.min(), maxPixels=65536
            )

            aggregation = "minimum observation count"

        elif group_name == "land_cover":

            continuous = [
                b for b in band_names
                if b != "dw_class_mode"
            ]

            probabilities = (
                image.select(continuous)
                .reduceResolution(
                    ee.Reducer.mean(), maxPixels=65536
                )
                .reproject(export_projection)
            )

            classes = (
                image.select("dw_class_mode")
                .reduceResolution(
                    ee.Reducer.mode(), maxPixels=65536
                )
                .reproject(export_projection)
            )

            image = probabilities.addBands(classes)

            aggregation = (
                "mean probabilities/statistics; mode class codes"
            )

        else:

            image = image.reduceResolution(
                ee.Reducer.mean(), maxPixels=65536
            )

        image = image.reproject(export_projection)

    elif group_name != "climate":

        aggregation = "nearest neighbour from fixed working grid"

    # Fill missing/outside-AOI cells and tag them as NoData.
    image = (
        image.select(band_names)
        .clip(aoi)
        .unmask(nodata_value, sameFootprint=False)
    )

    prepared = image.gd.prepareForExport(
        crs=output_crs,
        crs_transform=export_transform,
        shape=shape,
        dtype="float32",
        resampling="near"
    )

    filename = f"{group_name}.tif"
    destination = output_folder / filename

    print(
        f"Downloading {filename}: {len(band_names)} bands"
    )

    prepared.gd.toGeoTIFF(
        str(destination),
        overwrite=True,
        nodata=nodata_value,
        max_tile_size=8,
        max_requests=4
    )

    # Retain readable band names in each TIFF.
    with rasterio.open(destination, "r+") as raster:

        for index, band in enumerate(band_names, 1):

            raster.set_band_description(index, band)

            manifest.append({
                "file": filename,
                "group": group_name,
                "band_number": index,
                "band_name": band,
                "role": (
                    "QA" if group_name == "quality"
                    else "candidate_predictor"
                ),
                "data_type": (
                    "categorical" if band == "dw_class_mode"
                    else "numeric"
                ),
                "source": sources[group_name],
                "aggregation": aggregation
            })

# ---- Save the band manifest ----

with (output_folder / "band_manifest.csv").open(
    "w", newline=""
) as handle:

    writer = csv.DictWriter(
        handle, fieldnames=list(manifest[0])
    )
    writer.writeheader()
    writer.writerows(manifest)

# ---- Save settings and category definitions ----

settings = {
    "export_groups": list(export_groups),
    "analysis_scale": analysis_scale,
    "output_crs": output_crs,
    "export_cell_m": export_cell_m,
    "working_cell_m": working_cell_m,
    "export_transform": export_transform,
    "shape_rows_columns": shape,
    "context_radii_m": context_radii_m,
    "start_year": start_year,
    "end_year": end_year,
    "spring_months": [4, 6],
    "summer_months": [6, 8],
    "radar_pass": radar_pass,
    "cloud_percent": cloud_percent,
    "minimum_observations": minimum_observations,
    "median_maxRaw": 10000,
    "nodata": nodata_value,
    "sources": sources,
    "water_reference_period": (
        "1984-2021; masked descriptors filled with zero"
    ),
    "water_distance_cap_m": water_distance_max_m,
    "climate_native_support": "approximately 11 km",
    "analysis_scale_role": (
        "run label; groups and radii are explicitly selected"
    ),
    "units": {
        "topography": (
            "slope in degrees; elevation, relative elevation "
            "and relief in metres"
        ),
        "vegetation": "unitless indices and their summaries",
        "land_cover": (
            "probabilities and their summaries; "
            "dw_class_mode is categorical"
        ),
        "radar": (
            "dB except vv_over_vh_linear bands: unitless ratios"
        ),
        "water": (
            "occurrence/recurrence: percent; seasonality: months; "
            "distance: metres"
        ),
        "climate": (
            "temperature: degrees C; annual precipitation: mm; "
            "freezing months: months per year"
        ),
        "quality": "minimum valid native-pixel observation counts"
    },
    "dw_classes": {
        "0": "water",
        "1": "trees",
        "2": "grass",
        "3": "flooded vegetation",
        "4": "crops",
        "5": "shrub and scrub",
        "6": "built",
        "7": "bare",
        "8": "snow and ice"
    }
}

(output_folder / "settings.json").write_text(
    json.dumps(settings, indent=2)
)

(output_folder / "aoi.geojson").write_text(
    json.dumps(aoi_data)
)

# ---- Package and download to your computer ----

zip_path = shutil.make_archive(
    str(output_folder), "zip", output_folder
)

print("Ready:", zip_path)

files.download(zip_path)

  0%|          |0/108 tiles [00:00<?]